In [1]:
import sys
from pathlib import Path

import duckdb
from IPython.display import display


working_directory = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        directory
        for directory in (working_directory, *working_directory.parents)
        if (directory / "src" / "data" / "features.py").is_file()
        and (directory / "pyproject.toml").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook from within the project directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_customers
from src.data.splits import load_split_views
from src.data.features import METADATA_COLUMNS, load_feature_view

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\antek\Customer_Chunrn_End_to_End


In [2]:
with duckdb.connect() as connection:
    load_customers(connection)
    load_split_views(connection)
    load_feature_view(connection)

    training_data = connection.sql("""
        SELECT *
        FROM baseline_features
        WHERE partition = 'train'
        ORDER BY customer_id
    """).df()

if training_data.empty:
    raise ValueError("No training customers were loaded.")

if not training_data["partition"].eq("train").all():
    raise ValueError("Non-training customers found in training data.")

print(f"Training customers loaded: {len(training_data):,}")

Training customers loaded: 700,000


In [3]:
training_ids = training_data["customer_id"].copy()
y_train = training_data["churn"].astype("int64").copy()

X_train = training_data.drop(
    columns=list(METADATA_COLUMNS)
).copy()

if training_ids.isna().any() or not training_ids.is_unique:
    raise ValueError("Training customer IDs must be non-null and unique.")

if y_train.isna().any() or set(y_train.unique()) != {0, 1}:
    raise ValueError("Expected both binary churn classes.")

if set(METADATA_COLUMNS).intersection(X_train.columns):
    raise ValueError("Metadata must not appear in model predictors.")

if not (
    X_train.index.equals(y_train.index)
    and X_train.index.equals(training_ids.index)
):
    raise ValueError("Predictors, labels, and IDs are not aligned.")

print(f"Predictors: {X_train.shape}")
print(f"Target: {y_train.shape}")
print(f"Training churn rate: {100 * y_train.mean():.3f}%")

print("\nCLASS COUNTS")
display(
    y_train.value_counts()
    .sort_index()
    .rename_axis("churn")
    .to_frame("customers")
)

print("\nPREDICTOR TYPES AND MISSINGNESS")
display(
    X_train.dtypes.astype(str)
    .rename("dtype")
    .to_frame()
    .assign(
        missing_values=X_train.isna().sum(),
        missing_percentage=100 * X_train.isna().mean(),
    )
    .round(3)
)

Predictors: (700000, 28)
Target: (700000,)
Training churn rate: 9.923%

CLASS COUNTS


,customers
churn,
0,630541
1,69459



PREDICTOR TYPES AND MISSINGNESS


,dtype,missing_values,missing_percentage
age,int64,0,0.000
gender,str,0,0.000
annual_income,float64,20969,2.996
education,str,0,0.000
marital_status,str,0,0.000
dependents,int64,0,0.000
tenure,int64,0,0.000
contract,str,0,0.000
payment_method,str,0,0.000
paperless_billing,bool,0,0.000


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


numeric_columns = [
    "age",
    "annual_income",
    "dependents",
    "tenure",
    "monthlycharges",
    "totalcharges",
    "num_services",
    "customer_satisfaction",
    "num_complaints",
    "num_service_calls",
    "late_payments",
    "avg_monthly_gb",
    "days_since_last_interaction",
    "credit_score",
]

categorical_columns = [
    "gender",
    "education",
    "marital_status",
    "contract",
    "payment_method",
]

binary_columns = [
    "paperless_billing",
    "has_phone_service",
    "has_internet_service",
    "has_online_security",
    "has_online_backup",
    "has_device_protection",
    "has_tech_support",
    "has_streaming_tv",
    "has_streaming_movies",
]

assigned_columns = (
    numeric_columns + categorical_columns + binary_columns
)

if len(assigned_columns) != len(set(assigned_columns)):
    raise ValueError("A predictor is assigned to multiple preprocessing groups.")

if (
    not X_train.columns.is_unique
    or set(assigned_columns) != set(X_train.columns)
):
    raise ValueError("Preprocessing groups do not match the predictor columns.")

if X_train[categorical_columns + binary_columns].isna().any().any():
    raise ValueError("Unexpected missing categorical or binary values.")

if not X_train[binary_columns].isin([0, 1]).all().all():
    raise ValueError("Binary predictors must contain only 0/1 or False/True.")

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=False,
                keep_empty_features=True,
            ),
        ),
        ("scaler", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_columns),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
            ),
            categorical_columns,
        ),
        ("binary", "passthrough", binary_columns),
    ],
    remainder="drop",
)

In [5]:
from sklearn.linear_model import LogisticRegression


baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                solver="lbfgs",
                C=1.0,
                max_iter=1000,
                class_weight=None,
            ),
        ),
    ]
)

baseline_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: fl

In [6]:
import warnings

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import StratifiedKFold, cross_validate


cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

scoring = {
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
    "neg_log_loss": "neg_log_loss",
    "neg_brier_score": "neg_brier_score",
}

models = {
    "dummy_prior": DummyClassifier(strategy="prior"),
    "logistic_baseline": baseline_pipeline,
}

cv_results = {}
fold_tables = []

for model_name, model in models.items():
    print(f"Evaluating {model_name}...")

    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)

        result = cross_validate(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=1,
            error_score="raise",
            return_estimator=True,
        )

    cv_results[model_name] = result

    fold_tables.append(
        pd.DataFrame(
            {
                "model": model_name,
                "fold": range(1, cv.n_splits + 1),
                "average_precision": result["test_average_precision"],
                "roc_auc": result["test_roc_auc"],
                "log_loss": -result["test_neg_log_loss"],
                "brier_score": -result["test_neg_brier_score"],
                "fit_seconds": result["fit_time"],
            }
        )
    )

fold_metrics = pd.concat(fold_tables, ignore_index=True)

display(fold_metrics.round(4))

cv_summary = (
    fold_metrics.groupby("model")[
        [
            "average_precision",
            "roc_auc",
            "log_loss",
            "brier_score",
            "fit_seconds",
        ]
    ]
    .agg(["mean", "std"])
)

display(cv_summary.round(4))

Evaluating dummy_prior...
Evaluating logistic_baseline...


,model,fold,average_precision,roc_auc,log_loss,brier_score,fit_seconds
0,dummy_prior,1,0.0992,0.5000,0.3234,0.0894,0.0756
1,dummy_prior,2,0.0992,0.5000,0.3234,0.0894,0.0590
2,dummy_prior,3,0.0992,0.5000,0.3234,0.0894,0.0556
3,logistic_baseline,1,0.2024,0.6851,0.3031,0.0853,1.5630
4,logistic_baseline,2,0.2010,0.6846,0.3033,0.0853,1.5975
5,logistic_baseline,3,0.2008,0.6831,0.3034,0.0854,1.5722


average_precision         roc_auc        log_loss          \
                               mean     std    mean    std     mean     std   
model                                                                         
dummy_prior                  0.0992  0.0000  0.5000  0.000   0.3234  0.0000   
logistic_baseline            0.2014  0.0009  0.6843  0.001   0.3033  0.0002   

                  brier_score      fit_seconds          
                         mean  std        mean     std  
model                                                   
dummy_prior            0.0894  0.0      0.0634  0.0107  
logistic_baseline      0.0853  0.0      1.5776  0.0179

## Logistic Regression Baseline

Evaluated on 700,000 training customers using three-fold stratified
cross-validation (`random_state=42`).

**Pipeline:** median imputation and standard scaling for numeric features,
one-hot encoding for categorical features, and unchanged binary flags.
Logistic regression uses L2 regularization (`C=1.0`), without class
weighting, resampling, or interaction features. Preprocessing is fitted
within each fold.

| Metric (CV mean) | Dummy prior | Logistic regression |
|---|---:|---:|
| Average precision ↑ | 0.0992 | **0.2014** |
| ROC-AUC ↑ | 0.5000 | **0.6843** |
| Log loss ↓ | 0.3234 | **0.3033** |
| Brier score ↓ | 0.0894 | **0.0853** |

The model consistently improves on the dummy baseline. 

Validation and test data remain unused for model evaluation. These results
serve as the reference for subsequent pipeline improvements.

In [7]:
import numpy as np


fold_models = cv_results["logistic_baseline"]["estimator"]
fold_indices = list(cv.split(X_train, y_train))

if len(fold_models) != len(fold_indices):
    raise ValueError("Fold models and CV splits do not match.")

oof_probabilities = np.full(len(X_train), np.nan)
prediction_counts = np.zeros(len(X_train), dtype=int)

for model, (_, heldout_indices) in zip(
    fold_models, fold_indices, strict=True
):
    positive_class = np.flatnonzero(model.classes_ == 1)

    if len(positive_class) != 1:
        raise ValueError("Expected exactly one positive churn class.")

    probabilities = model.predict_proba(
        X_train.iloc[heldout_indices]
    )[:, positive_class[0]]

    oof_probabilities[heldout_indices] = probabilities
    prediction_counts[heldout_indices] += 1

if not np.all(prediction_counts == 1):
    raise ValueError("Each training customer must be scored exactly once.")

if (
    not np.isfinite(oof_probabilities).all()
    or (oof_probabilities < 0).any()
    or (oof_probabilities > 1).any()
):
    raise ValueError("Invalid out-of-fold probabilities.")

oof_predictions = pd.DataFrame(
    {
        "customer_id": training_ids.to_numpy(),
        "churn": y_train.to_numpy(),
        "probability": oof_probabilities,
    }
)

print(f"Out-of-fold predictions ready: {len(oof_predictions):,}")

Out-of-fold predictions ready: 700,000


In [9]:
import plotly.graph_objects as go
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    precision_recall_curve,
    roc_auc_score,
)


oof_metrics = pd.Series(
    {
        "average_precision": average_precision_score(
            y_train, oof_probabilities
        ),
        "roc_auc": roc_auc_score(y_train, oof_probabilities),
        "log_loss": log_loss(y_train, oof_probabilities),
        "brier_score": brier_score_loss(y_train, oof_probabilities),
    },
    name="pooled_oof",
)

display(oof_metrics.to_frame().round(4))

precision, recall, _ = precision_recall_curve(
    y_train, oof_probabilities
)

display_indices = np.unique(
    np.linspace(
        0,
        len(recall) - 1,
        min(2000, len(recall)),
        dtype=int,
    )
)

figure = go.Figure(
    go.Scatter(
        x=recall[display_indices],
        y=precision[display_indices],
        mode="lines",
        name="Logistic baseline",
    )
)

figure.add_hline(
    y=float(y_train.mean()),
    line_dash="dash",
    annotation_text="Dummy reference",
)

figure.update_layout(
    title="Training out-of-fold precision–recall curve",
    xaxis_title="Recall",
    yaxis_title="Precision",
    xaxis_range=[0, 1],
    yaxis_range=[0, 1],
)

figure.show()

,pooled_oof
average_precision,0.2013
roc_auc,0.6843
log_loss,0.3033
brier_score,0.0853


In [10]:
calibration_data = oof_predictions.copy()

calibration_data["risk_bin"] = pd.qcut(
    calibration_data["probability"],
    q=10,
    duplicates="drop",
)

calibration_table = (
    calibration_data.groupby("risk_bin", observed=True)
    .agg(
        customers=("churn", "size"),
        mean_probability=("probability", "mean"),
        observed_churn=("churn", "mean"),
    )
    .reset_index()
)

display(calibration_table.round(4))

figure = go.Figure()

figure.add_trace(
    go.Scatter(
        x=calibration_table["mean_probability"],
        y=calibration_table["observed_churn"],
        mode="lines+markers",
        name="Logistic baseline",
        customdata=calibration_table[["customers"]].to_numpy(),
        hovertemplate=(
            "Mean predicted risk: %{x:.2%}"
            "<br>Observed churn: %{y:.2%}"
            "<br>Customers: %{customdata[0]:,}"
            "<extra></extra>"
        ),
    )
)

axis_max = min(
    1.0,
    1.1 * max(
        calibration_table["mean_probability"].max(),
        calibration_table["observed_churn"].max(),
    ),
)

figure.add_trace(
    go.Scatter(
        x=[0, axis_max],
        y=[0, axis_max],
        mode="lines",
        line=dict(dash="dash"),
        name="Perfect calibration",
    )
)

figure.update_layout(
    title="Training out-of-fold calibration",
    xaxis_title="Mean predicted churn probability",
    yaxis_title="Observed churn share",
    xaxis=dict(range=[0, axis_max], tickformat=".0%"),
    yaxis=dict(range=[0, axis_max], tickformat=".0%"),
)

figure.show()

,risk_bin,customers,mean_probability,observed_churn
0,"(0.010599999999999998, 0.0359]",70000,0.0290,0.0293
1,"(0.0359, 0.0472]",70000,0.0415,0.0417
2,"(0.0472, 0.0589]",70000,0.0530,0.0518
3,"(0.0589, 0.0712]",70000,0.0650,0.0647
4,"(0.0712, 0.0844]",70000,0.0777,0.0775
5,"(0.0844, 0.0991]",70000,0.0915,0.0919
6,"(0.0991, 0.117]",70000,0.1077,0.1096
7,"(0.117, 0.141]",70000,0.1282,0.1301
8,"(0.141, 0.181]",70000,0.1588,0.1553
9,"(0.181, 0.783]",70000,0.2404,0.2404


In [11]:
ranked_customers = oof_predictions.sort_values(
    ["probability", "customer_id"],
    ascending=[False, True],
).reset_index(drop=True)

total_churned = int(ranked_customers["churn"].sum())
baseline_rate = ranked_customers["churn"].mean()

capture_rows = []

for fraction in (0.10, 0.20):
    selected_count = int(np.ceil(fraction * len(ranked_customers)))
    selected = ranked_customers.head(selected_count)

    captured = int(selected["churn"].sum())
    precision_at_budget = captured / selected_count

    capture_rows.append(
        {
            "top_percentage": 100 * fraction,
            "customers_selected": selected_count,
            "churned_customers_captured": captured,
            "precision_percentage": 100 * precision_at_budget,
            "capture_percentage": 100 * captured / total_churned,
            "lift": precision_at_budget / baseline_rate,
        }
    )

capture_table = pd.DataFrame(capture_rows)
display(capture_table.round(3))

,top_percentage,customers_selected,churned_customers_captured,precision_percentage,capture_percentage,lift
0,10.0,70000,16831,24.044,24.232,2.423
1,20.0,140000,27700,19.786,39.880,1.994


In [14]:
from sklearn.base import clone
from sklearn.preprocessing import FunctionTransformer


def short_tenure_flag(frame):
    return (
        (frame["tenure"] < 6)
        .astype(float)
        .to_numpy()
        .reshape(-1, 1)
    )


def short_monthly_flag(frame):
    return (
        (
            (frame["tenure"] < 6)
            & frame["contract"].eq("month_to_month")
        )
        .astype(float)
        .to_numpy()
        .reshape(-1, 1)
    )


def multiply_pair(values):
    return (values[:, 0] * values[:, 1]).reshape(-1, 1)


short_tenure_branch = (
    "short_tenure",
    FunctionTransformer(short_tenure_flag),
    ["tenure"],
)

short_monthly_branch = (
    "short_monthly",
    FunctionTransformer(short_monthly_flag),
    ["tenure", "contract"],
)

charges_satisfaction_branch = (
    "charges_satisfaction",
    Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    keep_empty_features=True,
                ),
            ),
            ("center_scale", StandardScaler()),
            ("product", FunctionTransformer(multiply_pair)),
            ("scale_product", StandardScaler()),
        ]
    ),
    ["monthlycharges", "customer_satisfaction"],
)

In [15]:
def pipeline_with_extra_features(extra_branches):
    candidate = clone(baseline_pipeline)
    candidate_preprocessor = candidate.named_steps["preprocessor"]

    candidate_preprocessor.set_params(
        transformers=[
            *candidate_preprocessor.transformers,
            *extra_branches,
        ]
    )

    return candidate


short_tenure_control = pipeline_with_extra_features(
    [short_tenure_branch]
)

interaction_pipeline = pipeline_with_extra_features(
    [
        short_tenure_branch,
        short_monthly_branch,
        charges_satisfaction_branch,
    ]
)

In [16]:
experimental_models = {
    "short_tenure_control": short_tenure_control,
    "selected_interactions": interaction_pipeline,
}

interaction_cv_results = {}
experiment_tables = []

for model_name, model in experimental_models.items():
    print(f"Evaluating {model_name}...")

    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)

        result = cross_validate(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=1,
            error_score="raise",
            return_estimator=True,
        )

    interaction_cv_results[model_name] = result

    experiment_tables.append(
        pd.DataFrame(
            {
                "model": model_name,
                "fold": range(1, cv.n_splits + 1),
                "average_precision": result["test_average_precision"],
                "roc_auc": result["test_roc_auc"],
                "log_loss": -result["test_neg_log_loss"],
                "brier_score": -result["test_neg_brier_score"],
                "fit_seconds": result["fit_time"],
            }
        )
    )

comparison_metrics = pd.concat(
    [
        fold_metrics.loc[
            fold_metrics["model"] == "logistic_baseline"
        ],
        *experiment_tables,
    ],
    ignore_index=True,
)

metric_columns = [
    "average_precision",
    "roc_auc",
    "log_loss",
    "brier_score",
]

display(comparison_metrics.round(5))

display(
    comparison_metrics.groupby("model")[metric_columns]
    .agg(["mean", "std"])
    .round(5)
)

Evaluating short_tenure_control...
Evaluating selected_interactions...


,model,fold,average_precision,roc_auc,log_loss,brier_score,fit_seconds
0,logistic_baseline,1,0.20237,0.68513,0.30309,0.08530,1.56300
1,logistic_baseline,2,0.20102,0.68460,0.30326,0.08535,1.59749
2,logistic_baseline,3,0.20077,0.68314,0.30341,0.08538,1.57223
3,short_tenure_control,1,0.20242,0.68516,0.30308,0.08529,1.55314
4,short_tenure_control,2,0.20104,0.68461,0.30326,0.08535,1.58353
5,short_tenure_control,3,0.20078,0.68312,0.30341,0.08538,1.58736
6,selected_interactions,1,0.20333,0.68563,0.30293,0.08525,1.74815
7,selected_interactions,2,0.20302,0.68499,0.30305,0.08528,1.68542
8,selected_interactions,3,0.20194,0.68349,0.30326,0.08533,1.68446


average_precision           roc_auc          log_loss  \
                                   mean      std     mean      std     mean   
model                                                                         
logistic_baseline               0.20139  0.00086  0.68429  0.00103  0.30325   
selected_interactions           0.20276  0.00073  0.68471  0.00110  0.30308   
short_tenure_control            0.20141  0.00088  0.68430  0.00105  0.30325   

                               brier_score           
                           std        mean      std  
model                                                
logistic_baseline      0.00016     0.08534  0.00004  
selected_interactions  0.00017     0.08529  0.00004  
short_tenure_control   0.00016     0.08534  0.00004

In [17]:
reference = (
    comparison_metrics.loc[
        comparison_metrics["model"] == "logistic_baseline"
    ]
    .set_index("fold")[metric_columns]
)

improvement_tables = []

for model_name in experimental_models:
    candidate = (
        comparison_metrics.loc[
            comparison_metrics["model"] == model_name
        ]
        .set_index("fold")[metric_columns]
    )

    improvement = candidate - reference
    improvement[["log_loss", "brier_score"]] *= -1
    improvement.insert(0, "model", model_name)

    improvement_tables.append(improvement.reset_index())

paired_improvements = pd.concat(
    improvement_tables,
    ignore_index=True,
)

display(paired_improvements.round(6))

,fold,model,average_precision,roc_auc,log_loss,brier_score
0,1,short_tenure_control,0.000044,0.000032,0.000009,0.000002
1,2,short_tenure_control,0.000023,0.000011,0.000003,0.000001
2,3,short_tenure_control,0.000008,-0.000011,-0.000002,-0.000000
3,1,selected_interactions,0.000958,0.000508,0.000166,0.000045
4,2,selected_interactions,0.002000,0.000395,0.000203,0.000072
5,3,selected_interactions,0.001168,0.000354,0.000145,0.000046


In [18]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer


def writable_numeric_array(frame):
    return np.array(frame, dtype=np.float64, copy=True)


imputation_bounds = {
    "age": (18, 90),
    "dependents": (0, 5),
    "tenure": (1, 72),
    "num_services": (1, 6),
    "customer_satisfaction": (1, 10),
    "num_complaints": (0, 8),
    "num_service_calls": (0, 12),
    "late_payments": (0, 5),
    "days_since_last_interaction": (1, 365),
    "credit_score": (300, 850),
}

minimum_values = np.array(
    [
        imputation_bounds.get(column, (0, np.inf))[0]
        for column in numeric_columns
    ]
)

maximum_values = np.array(
    [
        imputation_bounds.get(column, (0, np.inf))[1]
        for column in numeric_columns
    ]
)

iterative_numeric_pipeline = Pipeline(
    steps=[
        (
            "writable_array",
            FunctionTransformer(writable_numeric_array),
        ),
        (
            "imputer",
            IterativeImputer(
                initial_strategy="median",
                max_iter=20,
                tol=1e-3,
                n_nearest_features=8,
                skip_complete=True,
                sample_posterior=False,
                min_value=minimum_values,
                max_value=maximum_values,
                keep_empty_features=True,
                random_state=42,
            ),
        ),
        ("scaler", StandardScaler()),
    ]
)

iterative_pipeline = clone(baseline_pipeline)

iterative_pipeline.set_params(
    preprocessor__numeric=iterative_numeric_pipeline
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: fl

In [19]:
print("Evaluating iterative imputation...")

with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)

    iterative_cv_result = cross_validate(
        iterative_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        error_score="raise",
        return_estimator=True,
    )

iterative_fold_metrics = pd.DataFrame(
    {
        "model": "logistic_iterative",
        "fold": range(1, cv.n_splits + 1),
        "average_precision": iterative_cv_result[
            "test_average_precision"
        ],
        "roc_auc": iterative_cv_result["test_roc_auc"],
        "log_loss": -iterative_cv_result["test_neg_log_loss"],
        "brier_score": -iterative_cv_result["test_neg_brier_score"],
        "fit_seconds": iterative_cv_result["fit_time"],
    }
)

median_fold_metrics = fold_metrics.loc[
    fold_metrics["model"] == "logistic_baseline"
].copy()

imputation_comparison = pd.concat(
    [median_fold_metrics, iterative_fold_metrics],
    ignore_index=True,
)

display(imputation_comparison.round(5))

display(
    imputation_comparison.groupby("model")[
        [
            "average_precision",
            "roc_auc",
            "log_loss",
            "brier_score",
            "fit_seconds",
        ]
    ]
    .agg(["mean", "std"])
    .round(5)
)

Evaluating iterative imputation...


,model,fold,average_precision,roc_auc,log_loss,brier_score,fit_seconds
0,logistic_baseline,1,0.20237,0.68513,0.30309,0.08530,1.56300
1,logistic_baseline,2,0.20102,0.68460,0.30326,0.08535,1.59749
2,logistic_baseline,3,0.20077,0.68314,0.30341,0.08538,1.57223
3,logistic_iterative,1,0.20257,0.68519,0.30307,0.08529,4.21309
4,logistic_iterative,2,0.20112,0.68462,0.30325,0.08535,2.68712
5,logistic_iterative,3,0.20080,0.68317,0.30340,0.08537,2.25870


average_precision           roc_auc          log_loss  \
                                mean      std     mean      std     mean   
model                                                                      
logistic_baseline            0.20139  0.00086  0.68429  0.00103  0.30325   
logistic_iterative           0.20150  0.00094  0.68433  0.00104  0.30324   

                            brier_score          fit_seconds           
                        std        mean      std        mean      std  
model                                                                  
logistic_baseline   0.00016     0.08534  0.00004     1.57757  0.01786  
logistic_iterative  0.00017     0.08534  0.00004     3.05297  1.02727

In [20]:
imputation_improvements = (
    iterative_fold_metrics.set_index("fold")[metric_columns]
    - median_fold_metrics.set_index("fold")[metric_columns]
)

imputation_improvements[["log_loss", "brier_score"]] *= -1

display(imputation_improvements.round(6))

for fold, model in enumerate(
    iterative_cv_result["estimator"],
    start=1,
):
    fitted_imputer = (
        model.named_steps["preprocessor"]
        .named_transformers_["numeric"]
        .named_steps["imputer"]
    )

    print(
        f"Fold {fold}: "
        f"{fitted_imputer.n_iter_} imputation iterations"
    )

,average_precision,roc_auc,log_loss,brier_score
fold,,,,
1,0.000196,0.000058,0.000026,0.000008
2,0.000104,0.000023,0.000008,0.000003
3,0.000030,0.000038,0.000010,0.000003


Fold 1: 7 imputation iterations
Fold 2: 3 imputation iterations
Fold 3: 2 imputation iterations


In [21]:
from sklearn.ensemble import HistGradientBoostingClassifier


boosting_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", numeric_columns),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_columns,
        ),
        ("binary", "passthrough", binary_columns),
    ],
    remainder="drop",
    sparse_threshold=0,
)

boosting_pipeline = Pipeline(
    steps=[
        ("preprocessor", boosting_preprocessor),
        (
            "classifier",
            HistGradientBoostingClassifier(
                learning_rate=0.1,
                max_iter=150,
                max_leaf_nodes=15,
                min_samples_leaf=100,
                l2_regularization=1.0,
                early_stopping=False,
                random_state=42,
            ),
        ),
    ]
)

In [22]:
print("Evaluating histogram gradient boosting...")

with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)

    boosting_cv_result = cross_validate(
        boosting_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        error_score="raise",
        return_estimator=True,
    )

boosting_fold_metrics = pd.DataFrame(
    {
        "model": "hist_gradient_boosting",
        "fold": range(1, cv.n_splits + 1),
        "average_precision": boosting_cv_result[
            "test_average_precision"
        ],
        "roc_auc": boosting_cv_result["test_roc_auc"],
        "log_loss": -boosting_cv_result["test_neg_log_loss"],
        "brier_score": -boosting_cv_result["test_neg_brier_score"],
        "fit_seconds": boosting_cv_result["fit_time"],
    }
)

model_comparison = pd.concat(
    [
        median_fold_metrics,
        iterative_fold_metrics,
        boosting_fold_metrics,
    ],
    ignore_index=True,
)

display(model_comparison.round(5))

display(
    model_comparison.groupby("model")[
        [
            "average_precision",
            "roc_auc",
            "log_loss",
            "brier_score",
            "fit_seconds",
        ]
    ]
    .agg(["mean", "std"])
    .round(5)
)

Evaluating histogram gradient boosting...


,model,fold,average_precision,roc_auc,log_loss,brier_score,fit_seconds
0,logistic_baseline,1,0.20237,0.68513,0.30309,0.08530,1.56300
1,logistic_baseline,2,0.20102,0.68460,0.30326,0.08535,1.59749
2,logistic_baseline,3,0.20077,0.68314,0.30341,0.08538,1.57223
3,logistic_iterative,1,0.20257,0.68519,0.30307,0.08529,4.21309
4,logistic_iterative,2,0.20112,0.68462,0.30325,0.08535,2.68712
5,logistic_iterative,3,0.20080,0.68317,0.30340,0.08537,2.25870
6,hist_gradient_boosting,1,0.20421,0.68515,0.30294,0.08523,2.91425
7,hist_gradient_boosting,2,0.20360,0.68507,0.30300,0.08525,2.89364
8,hist_gradient_boosting,3,0.20264,0.68360,0.30325,0.08531,2.81695


average_precision           roc_auc          log_loss  \
                                    mean      std     mean      std     mean   
model                                                                          
hist_gradient_boosting           0.20348  0.00079  0.68461  0.00087  0.30306   
logistic_baseline                0.20139  0.00086  0.68429  0.00103  0.30325   
logistic_iterative               0.20150  0.00094  0.68433  0.00104  0.30324   

                                brier_score          fit_seconds           
                            std        mean      std        mean      std  
model                                                                      
hist_gradient_boosting  0.00016     0.08526  0.00004     2.87495  0.05127  
logistic_baseline       0.00016     0.08534  0.00004     1.57757  0.01786  
logistic_iterative      0.00017     0.08534  0.00004     3.05297  1.02727

In [23]:
boosting_improvements = (
    boosting_fold_metrics.set_index("fold")[metric_columns]
    - iterative_fold_metrics.set_index("fold")[metric_columns]
)

boosting_improvements[["log_loss", "brier_score"]] *= -1

display(boosting_improvements.round(6))

,average_precision,roc_auc,log_loss,brier_score
fold,,,,
1,0.001643,-0.000033,0.000125,0.000061
2,0.002478,0.000445,0.000251,0.000095
3,0.001838,0.000426,0.000148,0.000061


In [24]:
from time import perf_counter


candidate_cv_results = {
    "logistic_iterative": iterative_cv_result,
    "hist_gradient_boosting": boosting_cv_result,
}

candidate_oof = {}
runtime_rows = []
splits = list(cv.split(X_train, y_train))

if not (
    X_train.index.equals(y_train.index)
    and X_train.index.equals(training_ids.index)
):
    raise ValueError("Training rows, labels, and IDs are misaligned.")

for model_name, result in candidate_cv_results.items():
    probabilities = np.full(len(X_train), np.nan)
    prediction_counts = np.zeros(len(X_train), dtype=int)
    prediction_seconds = 0.0

    for model, (_, heldout_indices) in zip(
        result["estimator"], splits, strict=True
    ):
        positive_class = np.flatnonzero(model.classes_ == 1)

        if len(positive_class) != 1:
            raise ValueError("Expected exactly one positive churn class.")

        heldout_features = X_train.iloc[heldout_indices]

        started = perf_counter()
        fold_probabilities = model.predict_proba(
            heldout_features
        )[:, positive_class[0]]
        prediction_seconds += perf_counter() - started

        probabilities[heldout_indices] = fold_probabilities
        prediction_counts[heldout_indices] += 1

    if not np.all(prediction_counts == 1):
        raise ValueError("Each customer must have exactly one OOF prediction.")

    if (
        not np.isfinite(probabilities).all()
        or (probabilities < 0).any()
        or (probabilities > 1).any()
    ):
        raise ValueError(f"{model_name}: invalid probabilities.")

    candidate_oof[model_name] = pd.DataFrame(
        {
            "customer_id": training_ids.to_numpy(),
            "churn": y_train.to_numpy(),
            "probability": probabilities,
        }
    )

    runtime_rows.append(
        {
            "model": model_name,
            "mean_fold_fit_seconds": np.mean(result["fit_time"]),
            "oof_prediction_seconds": prediction_seconds,
            "batch_ms_per_1000_rows": (
                1000 * prediction_seconds / len(X_train) * 1000
            ),
        }
    )

runtime_comparison = pd.DataFrame(runtime_rows)
display(runtime_comparison.round(3))

,model,mean_fold_fit_seconds,oof_prediction_seconds,batch_ms_per_1000_rows
0,logistic_iterative,3.053,1.332,1.903
1,hist_gradient_boosting,2.875,1.545,2.207


In [25]:
targeting_rows = []

for model_name, predictions in candidate_oof.items():
    ranked = predictions.sort_values(
        ["probability", "customer_id"],
        ascending=[False, True],
    )

    total_churned = int(ranked["churn"].sum())
    baseline_rate = ranked["churn"].mean()

    if total_churned == 0:
        raise ValueError("Capture requires at least one churned customer.")

    for fraction in (0.10, 0.20):
        selected_count = int(np.ceil(fraction * len(ranked)))
        selected = ranked.head(selected_count)

        captured = int(selected["churn"].sum())
        precision_at_budget = captured / selected_count

        targeting_rows.append(
            {
                "model": model_name,
                "top_percentage": 100 * fraction,
                "customers_selected": selected_count,
                "churned_customers_captured": captured,
                "precision_percentage": 100 * precision_at_budget,
                "capture_percentage": 100 * captured / total_churned,
                "lift": precision_at_budget / baseline_rate,
            }
        )

targeting_comparison = pd.DataFrame(targeting_rows)
display(targeting_comparison.round(3))

targeting_differences = (
    targeting_comparison.loc[
        targeting_comparison["model"] == "hist_gradient_boosting"
    ]
    .set_index("top_percentage")[
        [
            "churned_customers_captured",
            "precision_percentage",
            "capture_percentage",
        ]
    ]
    - targeting_comparison.loc[
        targeting_comparison["model"] == "logistic_iterative"
    ]
    .set_index("top_percentage")[
        [
            "churned_customers_captured",
            "precision_percentage",
            "capture_percentage",
        ]
    ]
)

print("Boosting minus iterative logistic regression:")
display(targeting_differences.round(3))

,model,top_percentage,customers_selected,churned_customers_captured,precision_percentage,capture_percentage,lift
0,logistic_iterative,10.0,70000,16832,24.046,24.233,2.423
1,logistic_iterative,20.0,140000,27707,19.791,39.890,1.994
2,hist_gradient_boosting,10.0,70000,16958,24.226,24.414,2.441
3,hist_gradient_boosting,20.0,140000,27800,19.857,40.024,2.001


Boosting minus iterative logistic regression:


,churned_customers_captured,precision_percentage,capture_percentage
top_percentage,,,
10.0,126,0.180,0.181
20.0,93,0.066,0.134


In [26]:
calibration_tables = {}
figure = go.Figure()

for model_name, predictions in candidate_oof.items():
    data = predictions.copy()

    if data["probability"].nunique() < 2:
        raise ValueError(f"{model_name}: insufficient distinct risk scores.")

    data["risk_bin"] = pd.qcut(
        data["probability"],
        q=10,
        duplicates="drop",
    )

    table = (
        data.groupby("risk_bin", observed=True)
        .agg(
            customers=("churn", "size"),
            mean_probability=("probability", "mean"),
            observed_churn=("churn", "mean"),
        )
        .reset_index()
    )

    calibration_tables[model_name] = table

    print(f"\n{model_name}")
    display(table.round(4))

    figure.add_trace(
        go.Scatter(
            x=table["mean_probability"],
            y=table["observed_churn"],
            mode="lines+markers",
            name=model_name,
            customdata=table[["customers"]].to_numpy(),
            hovertemplate=(
                "Predicted: %{x:.2%}"
                "<br>Observed: %{y:.2%}"
                "<br>Customers: %{customdata[0]:,}"
                "<extra>%{fullData.name}</extra>"
            ),
        )
    )

axis_max = min(
    1.0,
    1.1 * max(
        table[["mean_probability", "observed_churn"]].to_numpy().max()
        for table in calibration_tables.values()
    ),
)

figure.add_trace(
    go.Scatter(
        x=[0, axis_max],
        y=[0, axis_max],
        mode="lines",
        name="Perfect calibration",
        line=dict(dash="dash", color="gray"),
    )
)

figure.update_layout(
    title="Training OOF calibration: candidate comparison",
    xaxis=dict(
        title="Mean predicted churn probability",
        range=[0, axis_max],
        tickformat=".0%",
    ),
    yaxis=dict(
        title="Observed churn share",
        range=[0, axis_max],
        tickformat=".0%",
    ),
)

figure.show()


logistic_iterative


,risk_bin,customers,mean_probability,observed_churn
0,"(0.010599999999999998, 0.0359]",70000,0.0289,0.0291
1,"(0.0359, 0.0471]",70000,0.0415,0.0416
2,"(0.0471, 0.0588]",70000,0.0529,0.0523
3,"(0.0588, 0.0711]",70000,0.0649,0.0644
4,"(0.0711, 0.0844]",70000,0.0777,0.0776
5,"(0.0844, 0.0991]",70000,0.0915,0.0920
6,"(0.0991, 0.117]",70000,0.1077,0.1094
7,"(0.117, 0.141]",70000,0.1282,0.1301
8,"(0.141, 0.181]",70000,0.1588,0.1554
9,"(0.181, 0.783]",70000,0.2405,0.2405



hist_gradient_boosting


,risk_bin,customers,mean_probability,observed_churn
0,"(0.0114, 0.0371]",70000,0.0312,0.0295
1,"(0.0371, 0.0471]",70000,0.0420,0.0411
2,"(0.0471, 0.0587]",70000,0.0528,0.0525
3,"(0.0587, 0.0712]",70000,0.0649,0.0657
4,"(0.0712, 0.0839]",70000,0.0775,0.0767
5,"(0.0839, 0.0979]",70000,0.0907,0.0920
6,"(0.0979, 0.115]",70000,0.1061,0.1090
7,"(0.115, 0.139]",70000,0.1263,0.1286
8,"(0.139, 0.18]",70000,0.1570,0.1549
9,"(0.18, 0.725]",70000,0.2439,0.2423


In [27]:
with duckdb.connect() as connection:
    load_customers(connection)
    load_split_views(connection)
    load_feature_view(connection)

    validation_data = connection.sql("""
        SELECT *
        FROM baseline_features
        WHERE partition = 'validation'
        ORDER BY customer_id
    """).df()

validation_ids = validation_data["customer_id"].copy()
y_validation = validation_data["churn"].copy()

X_validation = validation_data.drop(
    columns=list(METADATA_COLUMNS)
).copy()

if validation_data.empty:
    raise ValueError("Validation data is empty.")

if not validation_ids.is_unique or validation_ids.isna().any():
    raise ValueError("Validation IDs must be unique and non-null.")

if validation_ids.isin(training_ids).any():
    raise ValueError("Training and validation customers overlap.")

if y_validation.isna().any() or set(y_validation.unique()) != {0, 1}:
    raise ValueError("Expected both binary classes in validation.")

if (
    not X_validation.columns.equals(X_train.columns)
    or not X_validation.dtypes.equals(X_train.dtypes)
):
    raise ValueError("Training and validation feature schemas differ.")

print(f"Training: {len(X_train):,} customers")
print(f"Validation: {len(X_validation):,} customers")

Training: 700,000 customers
Validation: 150,000 customers


In [28]:
validation_candidates = {
    "logistic_iterative": iterative_pipeline,
    "hist_gradient_boosting": boosting_pipeline,
}

fitted_candidates = {}
validation_predictions = {}
validation_metric_rows = []

for model_name, pipeline in validation_candidates.items():
    print(f"Fitting {model_name} on all training customers...")

    model = clone(pipeline)

    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)

        started = perf_counter()
        model.fit(X_train, y_train)
        fit_seconds = perf_counter() - started

    positive_class = np.flatnonzero(model.classes_ == 1)

    if len(positive_class) != 1:
        raise ValueError("Expected exactly one positive churn class.")

    started = perf_counter()
    probabilities = model.predict_proba(
        X_validation
    )[:, positive_class[0]]
    prediction_seconds = perf_counter() - started

    if (
        not np.isfinite(probabilities).all()
        or (probabilities < 0).any()
        or (probabilities > 1).any()
    ):
        raise ValueError(f"{model_name}: invalid validation probabilities.")

    fitted_candidates[model_name] = model

    validation_predictions[model_name] = pd.DataFrame(
        {
            "customer_id": validation_ids.to_numpy(),
            "churn": y_validation.to_numpy(),
            "probability": probabilities,
        }
    )

    validation_metric_rows.append(
        {
            "model": model_name,
            "average_precision": average_precision_score(
                y_validation, probabilities
            ),
            "roc_auc": roc_auc_score(y_validation, probabilities),
            "log_loss": log_loss(y_validation, probabilities),
            "brier_score": brier_score_loss(
                y_validation, probabilities
            ),
            "fit_seconds": fit_seconds,
            "validation_prediction_seconds": prediction_seconds,
        }
    )

validation_metrics = pd.DataFrame(
    validation_metric_rows
).set_index("model")

display(validation_metrics.round(5))

Fitting logistic_iterative on all training customers...
Fitting hist_gradient_boosting on all training customers...


,average_precision,roc_auc,log_loss,brier_score,fit_seconds,validation_prediction_seconds
model,,,,,,
logistic_iterative,0.20251,0.68714,0.30279,0.08525,4.48294,0.27391
hist_gradient_boosting,0.20585,0.68846,0.30232,0.08511,4.08753,0.32596


In [29]:
validation_targeting_rows = []

for model_name, predictions in validation_predictions.items():
    ranked = predictions.sort_values(
        ["probability", "customer_id"],
        ascending=[False, True],
    )

    total_churned = int(ranked["churn"].sum())
    baseline_rate = ranked["churn"].mean()

    for fraction in (0.10, 0.20):
        selected_count = int(np.ceil(fraction * len(ranked)))
        captured = int(ranked.head(selected_count)["churn"].sum())
        precision_at_budget = captured / selected_count

        validation_targeting_rows.append(
            {
                "model": model_name,
                "top_percentage": 100 * fraction,
                "customers_selected": selected_count,
                "churned_customers_captured": captured,
                "precision_percentage": 100 * precision_at_budget,
                "capture_percentage": 100 * captured / total_churned,
                "lift": precision_at_budget / baseline_rate,
            }
        )

validation_targeting = pd.DataFrame(validation_targeting_rows)

display(validation_targeting.round(3))

,model,top_percentage,customers_selected,churned_customers_captured,precision_percentage,capture_percentage,lift
0,logistic_iterative,10.0,15000,3652,24.347,24.536,2.454
1,logistic_iterative,20.0,30000,6055,20.183,40.681,2.034
2,hist_gradient_boosting,10.0,15000,3696,24.640,24.832,2.483
3,hist_gradient_boosting,20.0,30000,6098,20.327,40.970,2.049


In [30]:
from sklearn.calibration import calibration_curve


figure = go.Figure()
validation_calibration_tables = {}

for model_name, predictions in validation_predictions.items():
    observed, predicted = calibration_curve(
        predictions["churn"],
        predictions["probability"],
        n_bins=10,
        strategy="quantile",
    )

    table = pd.DataFrame(
        {
            "mean_probability": predicted,
            "observed_churn": observed,
        }
    )

    validation_calibration_tables[model_name] = table

    print(f"\n{model_name}")
    display(table.round(4))

    figure.add_trace(
        go.Scatter(
            x=predicted,
            y=observed,
            mode="lines+markers",
            name=model_name,
        )
    )

axis_max = min(
    1.0,
    1.1 * max(
        table.to_numpy().max()
        for table in validation_calibration_tables.values()
    ),
)

figure.add_trace(
    go.Scatter(
        x=[0, axis_max],
        y=[0, axis_max],
        mode="lines",
        name="Perfect calibration",
        line=dict(dash="dash", color="gray"),
    )
)

figure.update_layout(
    title="Validation calibration — fixed candidates",
    xaxis=dict(
        title="Mean predicted churn probability",
        range=[0, axis_max],
        tickformat=".0%",
    ),
    yaxis=dict(
        title="Observed churn share",
        range=[0, axis_max],
        tickformat=".0%",
    ),
)

figure.show()


logistic_iterative


,mean_probability,observed_churn
0,0.0290,0.0311
1,0.0416,0.0397
2,0.0529,0.0503
3,0.0651,0.0639
4,0.0778,0.0765
5,0.0918,0.0923
6,0.1080,0.1044
7,0.1285,0.1303
8,0.1592,0.1602
9,0.2414,0.2435



hist_gradient_boosting


,mean_probability,observed_churn
0,0.0312,0.0306
1,0.0419,0.0398
2,0.0527,0.0518
3,0.0649,0.0644
4,0.0775,0.0732
5,0.0910,0.0915
6,0.1067,0.1061
7,0.1267,0.1283
8,0.1572,0.1601
9,0.2443,0.2464


## Selected Features and Missing-Value Strategy

**Model:** Histogram Gradient Boosting, which slightly
outperformed logistic regression in CV and validation.

- **Inputs:** 28 original predictors. Customer ID, churn target, and
  partition are excluded from model inputs; signup date and the
  inconsistent senior citizen flag are omitted.
- **Engineered features tested:** short tenure indicator, short tenure ×
  monthly contract, and standardized charges × satisfaction. Gains were
  small; these explicit features are **not included** in the selected
  boosting pipeline, which can learn nonlinearities and interactions.
- **Missing values:** no explicit imputation in the selected pipeline.
  Numeric nulls are handled natively by histogram gradient boosting.
- **Encoding:** categorical features are one-hot encoded; numeric and
  binary features pass through without scaling.

